## Imports & src definitions

All `src/` logic (UMF, User, create_dataloader, decentralized_train_n_epochs, etc.)
is inlined here so the notebook is fully self-contained.

In [2]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
from torch.optim import SGD
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import networkx as nx
import time
import math
import copy
import os

torch.manual_seed(0)
np.random.seed(0)

In [3]:
from os import chdir
from pathlib import Path

new_path = Path("/Users/haowen/Documents/Decentral RS/fed-learning-main")

if new_path.exists():
    os.chdir(new_path)
    print(f"Working directory changed to: {Path.cwd()}")
else:
    print("Path does not exist.")


Working directory changed to: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [4]:
from src.models.MatrixFactorization import MF, UMF
from src.graphs import random_k_out_graph, create_graph
from src.users import User
from src.training.decentralized import (decentralized_train_n_epochs,
                                        decentralized_validate_loop)
from src.data_utils import create_batched_dataloaders, create_dataloader

In [5]:
#Make data sample iterable
from torch.utils.data import Dataset, DataLoader, TensorDataset, Sampler
import torch
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim import SGD

from collections import Counter
import networkx as nx
from networkx.generators.classic import empty_graph
from networkx.utils import discrete_sequence, py_random_state, weighted_choice

import seaborn as sns
from sklearn.utils import shuffle

## Parameter Setting

In [7]:
model_type       = "umf"
val_loader_type  = "rs"
train_loader_type = "rs"
userprop         = None
n_factors        = 30
sparse           = False
batch_size       = 10
lr               = 0.03871364416669273
weight_decay     = 0.14214480688557163
mom              = 0.001
graph_seed       = 1
n_epochs         = 50
SEED             = 42

## Main

In [9]:
# ── Load sampled H&M dataset from cache ───────────────────────────────────────
# Set these to match what was used when sampling was run.
TARGET_USERS           = 1000
MIN_INTERACTIONS = 20
SAMPLED_DATA_DIR       = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/hm_sampled"

cache_tag  = f"u{TARGET_USERS}_m{MIN_INTERACTIONS}_s42"
train_path = os.path.join(SAMPLED_DATA_DIR, f"train_{cache_tag}.csv")
val_path   = os.path.join(SAMPLED_DATA_DIR, f"val_{cache_tag}.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, f"test_{cache_tag}.csv")
meta_path  = os.path.join(SAMPLED_DATA_DIR, f"meta_{cache_tag}.csv")

assert all(os.path.exists(p) for p in [train_path, val_path, test_path, meta_path]), (
    f"Cached files not found in '{SAMPLED_DATA_DIR}/'.\n"
    "Run the hm_foaf_experiment_sampled.ipynb preprocessing section first "
    "to generate the cache."
)

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)
meta_df  = pd.read_csv(meta_path)

n_users = int(meta_df.loc[meta_df["key"] == "n_users", "value"].iloc[0])
n_items = int(meta_df.loc[meta_df["key"] == "n_items", "value"].iloc[0])

print(f"Loaded from cache: {cache_tag}")
print(f"Total User: {n_users}")
print(f"Total Item: {n_items}")
train_df.head()

Loaded from cache: u1000_m20_s42
Total User: 948
Total Item: 7418


,customer_id,product_code,bought
0,115,339,4
1,634,546,1
2,630,230,2
3,233,222,2
4,778,9,1


In [10]:
# ── DataLoaders ───────────────────────────────────────────────────────────────
train_data_loader = create_dataloader(
    df=train_df, dl_type=train_loader_type, batch_size=batch_size, p=userprop
)
val_data_loader  = create_dataloader(df=val_df,  dl_type=val_loader_type)
test_data_loader = create_dataloader(df=test_df, dl_type=val_loader_type)

print(f"Train batches: {len(train_data_loader)} | "
      f"Val batches: {len(val_data_loader)} | "
      f"Test batches: {len(test_data_loader)}")

Train batches: 38708 | Val batches: 9677 | Test batches: 15696


In [11]:
# ── Initialise one UMF model per user ─────────────────────────────────────────
users = {}
for i in tqdm(range(n_users)):
    user_model = UMF(n_items, n_factors=n_factors, sparse=sparse)
    users[i] = User(
        id=i,
        model=user_model,
        optimizer=SGD(user_model.parameters(),
                      lr=lr, weight_decay=weight_decay, momentum=mom),
        model_name=model_type,
    )

  0%|          | 0/948 [00:00<?, ?it/s]

In [12]:
# ── Build graph ───────────────────────────────────────────────────────────────
graph = random_k_out_graph(n=n_users, k=2, seed=graph_seed)

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
torch.manual_seed(0)
train_losses, val_losses, time_per_epoch, commutes = decentralized_train_n_epochs(
    user_models=users,
    train_loader=train_data_loader,
    val_loader=val_data_loader,
    epochs=n_epochs,
    graph=graph,
)

  0%|          | 0/38708 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 1.9779 | Validation Loss: 1.4772 | Time Elapsed: 27.866383 sec |Commute: 77394 | Commute 
Cost: 8901214264

  0%|          | 0/38708 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 0.5116 | Validation Loss: 0.9282 | Time Elapsed: 38.764364 sec |Commute: 77394 | Commute 
Cost: 8901214264

  0%|          | 0/38708 [00:00<?, ?it/s]

...

  0%|          | 0/38708 [00:00<?, ?it/s]

Epoch 9 | Train Loss: 0.5690 | Validation Loss: 0.5035 | Time Elapsed: 31.680645 sec |Commute: 77394 | Commute 
Cost: 8901214264

  0%|          | 0/38708 [00:00<?, ?it/s]

Epoch 10 | Train Loss: 0.5732 | Validation Loss: 0.5035 | Time Elapsed: 21.875763 sec |Commute: 77394 | Commute 
Cost: 8901214264

  0%|          | 0/38708 [00:00<?, ?it/s]

In [ ]:
# ── Test evaluation ───────────────────────────────────────────────────────────
test_loss = decentralized_validate_loop(users, test_data_loader)

In [ ]:
test_loss